In [4]:
# ================================
# 📂 Cell 0 – Install & Upload Base Corn SnD File
# ================================
!pip install -q pandas numpy openpyxl statsmodels scikit-learn

from google.colab import files
import pandas as pd, numpy as np

print("📂 Please upload your CORN SnD Excel file (e.g. 'Corn SnD.xlsx')")
uploaded = files.upload()

xls_path = list(uploaded.keys())[0]
print(f"✅ File uploaded: {xls_path}")

# Load SnD sheet (adjust sheet_name if needed)
df = pd.read_excel(xls_path, sheet_name="SnD")
df.columns = df.columns.astype(str).str.strip()

# Expecting a 'Month' column
if "Month" not in df.columns:
    raise ValueError("❌ Missing 'Month' column in SnD sheet.")

# Convert Month → datetime, monthly start
df["date"] = pd.to_datetime(df["Month"], errors="coerce").dt.to_period("M").dt.to_timestamp()

# Treat 0 in numeric columns as missing
num_cols = df.select_dtypes(include=[np.number]).columns
df[num_cols] = df[num_cols].replace(0, np.nan)

# Build monthly time series
ts = df.sort_values("date").set_index("date").asfreq("MS")

print("🔎 Columns in SnD:")
print(list(ts.columns))
display(ts.tail(8))


📂 Please upload your CORN SnD Excel file (e.g. 'Corn SnD.xlsx')


Saving Corn.xlsx to Corn (1).xlsx
✅ File uploaded: Corn (1).xlsx
🔎 Columns in SnD:
['Month', 'Local Fees', 'Closing CBOT', 'Dollar Rate', 'Beginning Stock', 'Imports', 'Total Supply', 'Demand', 'Ending Stock', 'STU', 'Price ARG', 'Basis Average ARG', 'Replacement ARG', 'Price BRZ', 'Basis Average BRZ', 'Replacement BRZ', 'Price UKR', 'Basis Average UKR', 'Replacement UKR']


,Month,Local Fees,Closing CBOT,Dollar Rate,Beginning Stock,Imports,Total Supply,Demand,Ending Stock,STU,Price ARG,Basis Average ARG,Replacement ARG,Price BRZ,Basis Average BRZ,Replacement BRZ,Price UKR,Basis Average UKR,Replacement UKR
date,,,,,,,,,,,,,,,,,,,
2025-07-01,2025-07-01,510,403.403226,49.316000,403000,848000,1251000,966000,285000,0.295031,14080.65000,295.68,206.61,14080.65000,295.68,201.61,NaN,NaN,NaN
2025-08-01,2025-08-01,510,388.903226,48.528000,285000,782000,1067000,812000,255000,0.314039,14325.81000,334.32,223.94,14325.81000,334.32,218.94,NaN,NaN,NaN
2025-09-01,2025-09-01,510,422.437500,48.384000,255000,1454000,1709000,1193000,516000,0.432523,13150.00000,241.20,202.00,13150.00000,241.20,197.00,NaN,NaN,NaN
2025-10-01,2025-10-01,510,425.000000,47.600000,516000,1500000,2016000,1060000,956000,0.901887,11632.00000,188.49,190.42,11630.00000,188.49,185.42,NaN,240.90,172.48
2025-11-01,2025-11-01,510,427.951333,47.254333,956000,1230000,2186000,796000,1390000,1.746231,11767.38367,175.89,197.73,11768.82167,175.89,192.73,NaN,242.30,170.54
2025-12-01,2025-12-01,510,447.000000,47.600000,1388000,1306000,2694000,1025000,1669000,1.628293,11983.51000,165.07,203.23,11984.84000,203.23,198.23,NaN,240.98,159.74
2026-01-01,2026-01-26,510,426.000000,47.250000,1669000,1110000,2779000,1000000,1779000,1.779000,12200.00000,NaN,NaN,12200.00000,NaN,NaN,NaN,NaN,NaN
2026-02-01,2026-02-26,510,415.000000,47.100000,1779000,500000,2279000,1000000,1279000,1.279000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# ================================
# UPDATE for Cell 1 — Train with Tightness Features
# ================================
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV

# Forward/back fill key drivers & targets
for c in ["Closing CBOT", "Dollar Rate", "STU", "Price ARG", "Price BRZ"]:
    if c in ts.columns:
        ts[c] = pd.to_numeric(ts[c], errors="coerce").ffill().bfill()

hist = ts.dropna(subset=["Closing CBOT", "Dollar Rate", "STU", "Price ARG", "Price BRZ"]).copy()
print(f"✅ Usable monthly rows for modeling: {len(hist)}")

# ---- Tightness features (Removed for Cell 3 compatibility)
# hist["STU_z"] = (hist["STU"] - hist["STU"].mean()) / (hist["STU"].std(ddof=0) + 1e-9)
# hist["CBOT_x_Tight"] = hist["Closing CBOT"] * hist["STU_z"]

# (Optional) include STU change to capture tightening/loosening trend (Removed for Cell 3 compatibility)
# hist["dSTU"] = hist["STU"].diff().fillna(0)

# Feature matrix - changed to match the expected features in Cell 3
X_map = hist[["Closing CBOT", "Dollar Rate", "STU"]]

def fit_ridge(X, y):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("ridge",  RidgeCV(alphas=[0.1, 1.0, 10.0, 50.0]))
    ]).fit(X, y)

ridge_arg = fit_ridge(X_map, hist["Price ARG"])
ridge_brz = fit_ridge(X_map, hist["Price BRZ"])

print("✅ Trained monthly price models (with tightness features).")

✅ Usable monthly rows for modeling: 26
✅ Trained monthly price models (with tightness features).


In [6]:
# ================================
# 📅 Cell 2 – Upload DAILY CBOT + DAILY FX & Build merged_daily (FINAL + FX LOCK)
# ================================
from google.colab import files
import pandas as pd, numpy as np
import re
import os # Import os module to get file extension

print("📂 Please upload DAILY files:")
print("   1) CBOT daily file (OHLC or *_forecast_geo columns)")
print("   2) FX file (any sheet, any column names)")
uploaded = files.upload()

# -------------------------------------------------
# Helper functions
# -------------------------------------------------
def read_all_sheets(file):
    _, ext = os.path.splitext(file)
    if ext.lower() == '.csv':
        # For CSV, return a dictionary where the key is the basename and value is the DataFrame
        return {os.path.basename(file): pd.read_csv(file)}
    elif ext.lower() in ['.xlsx', '.xls']:
        return pd.read_excel(file, sheet_name=None)
    else:
        raise ValueError(f"Unsupported file type: {ext}")

def detect_date_col(df):
    candidates = []
    for c in df.columns:
        parsed = pd.to_datetime(df[c], errors="coerce")
        ok = parsed.notna().mean()
        bonus = 0.2 if any(k in str(c).lower() for k in ["date", "day", "time", "datetime", "timestamp"]) else 0
        if ok >= 0.5:
            candidates.append((ok + bonus, c))
    if not candidates:
        return None
    return sorted(candidates, reverse=True)[0][1]

def detect_numeric_col(df, exclude):
    rate_keys = ["rate", "fx", "usd", "dollar", "exchange", "egp", "sar", "aed", "eur", "gbp"]
    candidates = []
    for c in df.columns:
        if c in exclude:
            continue
        s = pd.to_numeric(df[c], errors="coerce")
        ok = s.notna().mean()
        if ok >= 0.6:
            bonus = 0.2 if any(k in str(c).lower() for k in rate_keys) else 0
            med = float(s.dropna().median()) if s.notna().any() else np.nan
            plaus = 0.1 if np.isfinite(med) and (0.1 <= med <= 500) else 0.0
            candidates.append((ok + bonus + plaus, c))
    if not candidates:
        return None
    return sorted(candidates, reverse=True)[0][1]

def detect_ohlc(df):
    def f(pats):
        for c in df.columns:
            for p in pats:
                if re.search(p, str(c), re.I):
                    return c
        return None
    return {
        "Open":  f([r"\bopen\b"]),
        "High":  f([r"\bhigh\b"]),
        "Low":   f([r"\blow\b"]),
        "Close": f([r"\bclose\b", r"settle", r"last"])
    }

def detect_forecast_geo(df):
    high = next((c for c in df.columns if "high_forecast_geo"  in str(c).lower()), None)
    mid  = next((c for c in df.columns if "price_forecast_geo" in str(c).lower()), None)
    low  = next((c for c in df.columns if "low_forecast_geo"   in str(c).lower()), None)
    return high, mid, low

# -------------------------------------------------
# Read all files
# -------------------------------------------------
books = {f: read_all_sheets(f) for f in uploaded.keys()}

# -------------------------------------------------
# Detect CBOT
# -------------------------------------------------
best_cbot = None
for fname, book in books.items():
    for sh, df in book.items():
        df = df.copy()
        df.columns = df.columns.astype(str).str.strip()
        if df.shape[1] < 2 or df.shape[0] < 3:
            continue

        date_col = detect_date_col(df)
        if not date_col:
            continue

        high_f, mid_f, low_f = detect_forecast_geo(df)
        ohlc = detect_ohlc(df)

        mode = None
        cols = None
        score = 0.0

        if mid_f:
            mode = "forecast"
            cols = (high_f, mid_f, low_f)
            score = 3.0
        elif ohlc["Close"]:
            mode = "ohlc"
            cols = ohlc
            score = 2.0
        else:
            continue

        date_ok = pd.to_datetime(df[date_col], errors="coerce").notna().mean()
        score += date_ok

        if not best_cbot or score > best_cbot[0]:
            best_cbot = (score, fname, sh, df, date_col, mode, cols)

if not best_cbot:
    raise RuntimeError("❌ CBOT not detected")

_, cbot_file, cbot_sheet, df, date_col, mode, cols = best_cbot
print(f"✅ CBOT detected from: {cbot_file} | sheet '{cbot_sheet}' | mode={mode}")

df["Date"] = pd.to_datetime(df[date_col], errors="coerce").dt.normalize()
df = df.dropna(subset=["Date"]).sort_values("Date").set_index("Date")

cbot = pd.DataFrame(index=df.index)

if mode == "forecast":
    high_f, mid_f, low_f = cols
    cbot["CBOT_Close"] = pd.to_numeric(df[mid_f], errors="coerce")
    cbot["CBOT_Low"]   = pd.to_numeric(df[low_f], errors="coerce") if low_f else cbot["CBOT_Close"]
    cbot["CBOT_High"]  = pd.to_numeric(df[high_f], errors="coerce") if high_f else cbot["CBOT_Close"]
    cbot["CBOT_Open"]  = cbot["CBOT_Close"]
else:
    cbot["CBOT_Close"] = pd.to_numeric(df[cols["Close"]], errors="coerce")
    cbot["CBOT_Open"]  = pd.to_numeric(df[cols["Open"]], errors="coerce") if cols["Open"] else cbot["CBOT_Close"]
    cbot["CBOT_High"]  = pd.to_numeric(df[cols["High"]], errors="coerce") if cols["High"] else cbot["CBOT_Close"]
    cbot["CBOT_Low"]   = pd.to_numeric(df[cols["Low"]], errors="coerce") if cols["Low"] else cbot["CBOT_Close"]

cbot = cbot.dropna(subset=["CBOT_Close"])
cbot["CBOT_Low"]  = cbot["CBOT_Low"].fillna(cbot["CBOT_Close"])
cbot["CBOT_High"] = cbot["CBOT_High"].fillna(cbot["CBOT_Close"])
cbot["CBOT_Open"] = cbot["CBOT_Open"].fillna(cbot["CBOT_Close"])
cbot["Closing CBOT"] = cbot["CBOT_Close"]  # backward compatible

# Handle duplicates after normalize
if cbot.index.duplicated().any():
    print("⚠️ CBOT has duplicate dates — keeping last per date")
    cbot = cbot.groupby(level=0).last()

# -------------------------------------------------
# Detect FX
# -------------------------------------------------
best_fx = None
for fname, book in books.items():
    for sh, df in book.items():
        df = df.copy()
        df.columns = df.columns.astype(str).str.strip()
        if df.shape[1] < 2 or df.shape[0] < 3:
            continue

        date_col = detect_date_col(df)
        if not date_col:
            continue

        rate_col = detect_numeric_col(df, exclude=[date_col])
        if not rate_col:
            continue

        # Simple score: date parse + numeric parse
        date_ok = pd.to_datetime(df[date_col], errors="coerce").notna().mean()
        rate_ok = pd.to_numeric(df[rate_col], errors="coerce").notna().mean()
        score = date_ok + rate_ok

        if not best_fx or score > best_fx[0]:
            best_fx = (score, fname, sh, df, date_col, rate_col)

if not best_fx:
    raise RuntimeError("❌ FX not detected")

_, fx_file, fx_sheet, df, date_col, rate_col = best_fx
print(f"✅ FX detected from: {fx_file} | sheet '{fx_sheet}' | Date='{date_col}' | Rate='{rate_col}'")

df["Date"] = pd.to_datetime(df[date_col], errors="coerce").dt.normalize()
df["Dollar Rate"] = pd.to_numeric(df[rate_col], errors="coerce")
fx = df.dropna(subset=["Date", "Dollar Rate"]).sort_values("Date").set_index("Date")[["Dollar Rate"]]

# Handle duplicates after normalize
if fx.index.duplicated().any():
    print("⚠️ FX has duplicate dates — averaging per date")
    fx = fx.groupby(level=0)[["Dollar Rate"]].mean()

# -------------------------------------------------
# FX OPTION 2 – SCALE TO TODAY + LOCK AFTER LAST DATE
# -------------------------------------------------
TARGET_FX = 47.1

last_fx_date  = fx.index.max()
last_fx_value = float(fx.loc[last_fx_date, "Dollar Rate"])

scale = TARGET_FX / last_fx_value
fx["Dollar Rate"] = fx["Dollar Rate"] * scale

print(f"⚠️ FX scaled from {last_fx_value:.2f} ({last_fx_date.date()}) → {TARGET_FX}")

# 🔒 LOCK FX after last historical date (flat)
last_fx_value_scaled = float(fx.loc[last_fx_date, "Dollar Rate"])
fx_lock_value = last_fx_value_scaled  # should equal TARGET_FX

# We'll lock later after we build the full daily index (so it affects future dates too)
print(f"🔒 FX will be locked at {fx_lock_value:.2f} for dates AFTER {last_fx_date.date()}")

# -------------------------------------------------
# Merge daily
# -------------------------------------------------
start = min(cbot.index.min(), fx.index.min())
end   = max(cbot.index.max(), fx.index.max())

idx = pd.date_range(start=start, end=end, freq="D")

merged_daily = pd.DataFrame(index=idx)

for c in ["CBOT_Low", "CBOT_High", "CBOT_Open", "CBOT_Close", "Closing CBOT"]:
    merged_daily[c] = cbot[c].reindex(idx).ffill().bfill()

merged_daily["Dollar Rate"] = fx["Dollar Rate"].reindex(idx).ffill().bfill()

# 🔒 Apply lock in merged_daily for any date after last_fx_date
merged_daily.loc[merged_daily.index > last_fx_date, "Dollar Rate"] = fx_lock_value

print("✅ merged_daily READY (with CBOT scenarios + FX scaled + FX locked)")
display(merged_daily.tail(10))


📂 Please upload DAILY files:
   1) CBOT daily file (OHLC or *_forecast_geo columns)
   2) FX file (any sheet, any column names)


Saving Corn_Forecast_Jan_Feb_2026_REALISTIC.csv to Corn_Forecast_Jan_Feb_2026_REALISTIC (1).csv
Saving Exchange Rates Historical.xlsx to Exchange Rates Historical (1).xlsx
✅ CBOT detected from: Corn_Forecast_Jan_Feb_2026_REALISTIC (1).csv | sheet 'Corn_Forecast_Jan_Feb_2026_REALISTIC (1).csv' | mode=ohlc
✅ FX detected from: Corn_Forecast_Jan_Feb_2026_REALISTIC (1).csv | sheet 'Corn_Forecast_Jan_Feb_2026_REALISTIC (1).csv' | Date='Date' | Rate='Open'
⚠️ FX scaled from 401.64 (2026-02-27) → 47.1
🔒 FX will be locked at 47.10 for dates AFTER 2026-02-27
✅ merged_daily READY (with CBOT scenarios + FX scaled + FX locked)


/tmp/ipython-input-301696663.py:30: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(df[c], errors="coerce")
/tmp/ipython-input-301696663.py:30: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(df[c], errors="coerce")
/tmp/ipython-input-301696663.py:30: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(df[c], errors="coerce")
/tmp/ipython-input-301696663.py:30: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expec

,CBOT_Low,CBOT_High,CBOT_Open,CBOT_Close,Closing CBOT,Dollar Rate
2026-02-18,403.501721,412.086846,408.229703,407.358864,407.358864,47.872610
2026-02-19,402.616434,411.216006,407.358864,406.473577,406.473577,47.770488
2026-02-20,402.152199,410.330720,406.473577,406.009342,406.009342,47.666672
2026-02-21,402.152199,410.330720,406.473577,406.009342,406.009342,47.666672
2026-02-22,402.152199,410.330720,406.473577,406.009342,406.009342,47.666672
2026-02-23,401.140612,409.866485,406.009342,404.997755,404.997755,47.612231
2026-02-24,400.012377,408.854898,404.997755,403.869520,403.869520,47.493604
2026-02-25,398.924215,407.726663,403.869520,402.781358,402.781358,47.361297
2026-02-26,397.784189,406.638501,402.781358,401.641332,401.641332,47.233689
2026-02-27,397.103622,405.498475,401.641332,400.960765,400.960765,47.100000


In [7]:
# ================================
# 🔮 Cell 3 – FULL-MONTH Daily Prices (CBOT Scenarios, FX FIXED)
# ================================
import numpy as np
import pandas as pd

# -----------------------------
# Safety checks
# -----------------------------
for r in ["merged_daily", "ridge_arg", "ridge_brz", "ts"]:
    if r not in globals():
        raise NameError(f"❌ Missing variable: {r}. Run previous cells first.")

# ---------------------------------------------------------
# 1) Choose forecast month
# ---------------------------------------------------------
forecast_input = input("📅 Enter forecast month (YYYY-MM), e.g. 2026-02: ").strip()
forecast_period = pd.Period(forecast_input, freq="M")

year  = forecast_period.year
month = forecast_period.month
days_in_month = forecast_period.days_in_month

full_month_idx = pd.date_range(
    start=f"{year}-{month:02d}-01",
    end=f"{year}-{month:02d}-{days_in_month}",
    freq="D"
)

# ---------------------------------------------------------
# 2) Extract daily CBOT & FX for the forecast month
# ---------------------------------------------------------
window = merged_daily.reindex(full_month_idx).copy()
window = window.bfill().ffill()
window.index.name = "Date"

# ---------------------------------------------------------
# 🔒 FIX FX – keep it constant for the forecast month
# ---------------------------------------------------------
TARGET_FX = 47.1
window["Dollar Rate"] = TARGET_FX

print(f"\n🔒 FX fixed at {TARGET_FX} for the whole forecast month")

# Make sure CBOT scenario columns exist
for col in ["CBOT_Low", "CBOT_High", "CBOT_Open", "CBOT_Close"]:
    if col not in window.columns:
        window[col] = window["Closing CBOT"]

if "Closing CBOT" not in window.columns and "CBOT_Close" in window.columns:
    window["Closing CBOT"] = window["CBOT_Close"]

print(f"\n📆 Forecast month selected: {forecast_period}")
print(f"📅 Total days: {len(window)}")

# ---------------------------------------------------------
# 3) User inputs: Imports & Demand
# ---------------------------------------------------------
print("\n🔢 Enter your SnD assumptions for this month:")
imports_input = float(input("🚢 Expected monthly IMPORTS (tons): "))
demand_input  = float(input("📊 Expected monthly DEMAND (tons): "))

# ---------------------------------------------------------
# 4) Compute STU
# ---------------------------------------------------------
if "Ending Stock" in ts.columns and ts["Ending Stock"].notna().any():
    begin_stock = float(ts["Ending Stock"].dropna().iloc[-1])
elif "Beginning Stock" in ts.columns and ts["Beginning Stock"].notna().any():
    begin_stock = float(ts["Beginning Stock"].dropna().iloc[-1])
else:
    begin_stock = 0.3 * demand_input  # fallback

end_stock = begin_stock + imports_input - demand_input
stu_forecast = max(end_stock / demand_input, 0)

print("\n📦 STOCKS SUMMARY:")
print(f"   Beginning Stock ≈ {begin_stock:,.0f}")
print(f"   Ending Stock    ≈ {end_stock:,.0f}")
print(f"   STU             ≈ {stu_forecast:.2f}")

# ---------------------------------------------------------
# Helper: SAME pricing logic as before (only CBOT scenario differs)
# ---------------------------------------------------------
def scenario_forecast(window_df: pd.DataFrame, cbot_col: str, tag: str):
    w = window_df.copy()

    # Monthly means
    cbot_mean = float(w[cbot_col].mean())
    fx_mean   = float(w["Dollar Rate"].mean())  # fixed, but kept for structure

    # Monthly model input (UNCHANGED)
    X_month = pd.DataFrame({
        "Closing CBOT": [cbot_mean],
        "Dollar Rate":  [fx_mean],
        "STU":          [stu_forecast]
    })

    arg_month = float(ridge_arg.predict(X_month)[0])
    brz_month = float(ridge_brz.predict(X_month)[0])

    # Daily deviations (UNCHANGED)
    w["CBOT_DEV"] = (w[cbot_col] - cbot_mean) / (cbot_mean if cbot_mean != 0 else 1)
    w["FX_DEV"]   = 0.0  # FX fixed → no daily FX noise

    cbot_ret = w[cbot_col].pct_change()
    vol_cbot = float(cbot_ret.std(skipna=True) or 0)

    # Same weights as before
    cbot_weight = 0.70
    fx_weight   = 0.30

    raw_mod = 1 + (cbot_weight * w["CBOT_DEV"])
    band = 0.03 + min(0.04, 2 * vol_cbot)
    w["Modifier"] = raw_mod.clip(1 - band, 1 + band)

    out = pd.DataFrame(index=w.index)
    out[f"CBOT ({tag})"] = w[cbot_col]
    out[f"ARG Daily Price ({tag})"] = arg_month * w["Modifier"]
    out[f"BRZ Daily Price ({tag})"] = brz_month * w["Modifier"]
    return out

# ---------------------------------------------------------
# 5) Build the 4 CBOT scenarios
# ---------------------------------------------------------
scenario_defs = [
    ("Low Futures",   "CBOT_Low"),
    ("High Futures",  "CBOT_High"),
    ("Open Futures",  "CBOT_Open"),
    ("Close Futures", "CBOT_Close"),
]

frames = [scenario_forecast(window, col, tag) for tag, col in scenario_defs]
scenario_out = pd.concat(frames, axis=1)

# Add FX column for visibility
scenario_out["Dollar Rate"] = window["Dollar Rate"]

# Nice column order
ordered = ["Dollar Rate"]
for tag, _ in scenario_defs:
    ordered += [f"CBOT ({tag})", f"ARG Daily Price ({tag})", f"BRZ Daily Price ({tag})"]
scenario_out = scenario_out[ordered]

print("\n✅ Scenario forecast complete (4 CBOT views, FX fixed).")
display(scenario_out.head())
display(scenario_out.tail())

# ---------------------------------------------------------
# Backward compatibility (same outputs as before)
# ---------------------------------------------------------
for c in scenario_out.columns:
    window[c] = scenario_out[c]

window["ARG Daily Price"] = window["ARG Daily Price (Close Futures)"]
window["BRZ Daily Price"] = window["BRZ Daily Price (Close Futures)"]


📅 Enter forecast month (YYYY-MM), e.g. 2026-02: 2026-2

🔒 FX fixed at 47.1 for the whole forecast month

📆 Forecast month selected: 2026-02
📅 Total days: 28

🔢 Enter your SnD assumptions for this month:
🚢 Expected monthly IMPORTS (tons): 500000
📊 Expected monthly DEMAND (tons): 1000000

📦 STOCKS SUMMARY:
   Beginning Stock ≈ 1,279,000
   Ending Stock    ≈ 779,000
   STU             ≈ 0.78

✅ Scenario forecast complete (4 CBOT views, FX fixed).


,Dollar Rate,CBOT (Low Futures),ARG Daily Price (Low Futures),BRZ Daily Price (Low Futures),CBOT (High Futures),ARG Daily Price (High Futures),BRZ Daily Price (High Futures),CBOT (Open Futures),ARG Daily Price (Open Futures),BRZ Daily Price (Open Futures),CBOT (Close Futures),ARG Daily Price (Close Futures),BRZ Daily Price (Close Futures)
Date,,,,,,,,,,,,,
2026-02-01,47.1,417.832870,12269.344587,12270.025003,426.959770,12314.566554,12315.221469,423.102628,12298.686001,12299.352503,421.690012,12285.255813,12285.924648
2026-02-02,47.1,417.306303,12258.430874,12259.110685,425.547155,12285.804379,12286.457764,421.690012,12269.696808,12270.361739,421.163446,12274.428728,12275.096974
2026-02-03,47.1,414.466349,12199.569491,12200.246038,425.020589,12275.082987,12275.735802,421.163446,12258.890793,12259.555138,418.323492,12216.034556,12216.699623
2026-02-04,47.1,412.817161,12165.388128,12166.062779,422.180635,12217.258855,12217.908595,418.323492,12200.610259,12201.271446,416.674304,12182.124507,12182.787727
2026-02-05,47.1,412.245973,12153.549583,12154.223578,420.531447,12183.679833,12184.327787,416.674304,12166.766201,12167.425553,416.103116,12170.379931,12171.042512


,Dollar Rate,CBOT (Low Futures),ARG Daily Price (Low Futures),BRZ Daily Price (Low Futures),CBOT (High Futures),ARG Daily Price (High Futures),BRZ Daily Price (High Futures),CBOT (Open Futures),ARG Daily Price (Open Futures),BRZ Daily Price (Open Futures),CBOT (Close Futures),ARG Daily Price (Close Futures),BRZ Daily Price (Close Futures)
Date,,,,,,,,,,,,,
2026-02-24,47.1,400.012377,11899.993911,11900.653844,408.854898,11945.934298,11946.569609,404.997755,11927.144160,11927.790526,403.869520,11918.836852,11919.485739
2026-02-25,47.1,398.924215,11877.440461,11878.099144,407.726663,11922.962371,11923.596460,403.869520,11903.990916,11904.636028,402.781358,11896.462420,11897.110089
2026-02-26,47.1,397.784189,11853.812084,11854.469456,406.638501,11900.806356,11901.439266,402.781358,11881.660026,11882.303927,401.641332,11873.021593,11873.667986
2026-02-27,47.1,397.103622,11839.706525,11840.363115,405.498475,11877.594357,11878.226032,401.641332,11858.264816,11858.907450,400.960765,11859.027997,11859.673628
2026-02-28,47.1,397.103622,11839.706525,11840.363115,405.498475,11877.594357,11878.226032,401.641332,11858.264816,11858.907450,400.960765,11859.027997,11859.673628


In [8]:
# ================================
# 💾 Cell 4 – Save daily prices to Excel
# ================================
output_path = f"Corn_Daily_Prices_with_SnD_{str(forecast_period)}.xlsx"
window.round(2).to_excel(output_path)
print(f"📁 Saved daily prices to: {output_path}")

from google.colab import files
files.download(output_path)


📁 Saved daily prices to: Corn_Daily_Prices_with_SnD_2026-02.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>